In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from catboost import CatBoostRegressor

from var import DATA_OUT
from scintill_ai.preprocess import get_time_filtering_and_features

In [ ]:
# Load pre-processed dataset
df = pd.read_parquet(Path(DATA_OUT, 'df.parquet'), engine='pyarrow').drop(
    columns=[ 's4_max']
)

# Filter by time window (from 20 to 06 UTC) and create temporal features (lag and MAVs)
df = get_time_filtering_and_features(
    df, hour_start=20, hour_stop=6, ema_cols={'s4_mean': [5]}, lag_cols={'h_tmk': [180]},
)

# Target
df['s4_mean_lead'] = df['s4_mean'].shift(-5)

In [ ]:
# df.loc['2022-01-01':'2023-01-31',['s4_mean','h_tmk']].isna().sum() / df.loc['2022-01-01':'2023-01-31'].shape[0]

In [ ]:
TRAIN_START, TRAIN_STOP = '2021-12-01', '2023-01-31'
CALIB_START, CALIB_STOP = '2023-03-01', '2023-03-17'
TEST_START, TEST_STOP = '2023-03-18', '2023-03-31'

In [ ]:
X_cols = [
    'n_sat',
    's4_mean',
    'field_magnitude_avg',
    'wind_speed',
    'wind_density',
    'wind_pressure',
    'eletric_field',
    'h_tmk',
    'f10.7_adj',
    'sza',
    's4_mean_variation_online',
    's4_mean_ema_5m',
    'h_tmk_lag_180m',
]

y_col = 's4_mean_lead'

X_train, X_calib, X_test = (
    df.loc[TRAIN_START:TRAIN_STOP, X_cols].copy(),
    df.loc[CALIB_START:CALIB_STOP, X_cols].copy(),
    df.loc[TEST_START:TEST_STOP, X_cols].copy(),
)
y_train, y_calib, y_test = (
    df.loc[TRAIN_START:TRAIN_STOP, y_col].copy().fillna(0),
    df.loc[CALIB_START:CALIB_STOP, y_col].copy().fillna(0),
    df.loc[TEST_START:TEST_STOP, y_col].copy().fillna(0),
)

In [ ]:
# tscv = TimeSeriesSplit(n_splits=3)

# model = CatBoostRegressor(
#     loss_function="RMSE",
#     cat_features=['s4_mean_variation_online'],
#     random_seed=42,
#     thread_count=-1,
#     bootstrap_type="Bernoulli",
#     sampling_frequency='PerTree',
#     od_type="Iter",
#     od_wait=300,
#     eval_metric="RMSE",
#     use_best_model=True,
#     subsample=0.9,
#     colsample_bylevel=0.9,
#     has_time=True,
# )

# param_grid = {
#     "iterations": [500, 1_000],
#     "max_depth": [4, 6],
#     "l2_leaf_reg": [25, 40],
#     "min_data_in_leaf": [15, 30],
# }

# grid_search = GridSearchCV(
#     estimator=model,
#     param_grid=param_grid,
#     cv=tscv,
#     scoring="neg_root_mean_squared_error",
#     n_jobs=-1,
#     verbose=1,
# )

# grid_search.fit(X_train, y_train, eval_set=(X_test, y_test))

# grid_search.best_params_
# # {'iterations': 500, 'l2_leaf_reg': 30, 'max_depth': 4, 'min_data_in_leaf': 50}

In [ ]:
cb_params = {
    'iterations': 141,
    'l2_leaf_reg': 25,
    'max_depth': 4,
    'min_data_in_leaf': 15,
    # 'cat_features': ['s4_mean_variation_online'],
    'random_seed': 42,
    'thread_count': -1,
    'bootstrap_type': "Bernoulli",
    'sampling_frequency': 'PerTree',
    'subsample': 0.9,
    'colsample_bylevel': 0.9,
    'has_time': True,
}

## CatBoost + ACI

In [ ]:
from mapie.subsample import BlockBootstrap

from scintill_ai.conformal import aci_ts_regressor_predict
from scintill_ai import LW_S4_THRESHOLD

In [ ]:
cv = BlockBootstrap(
    n_resamplings=10, n_blocks=10, overlapping=False, random_state=42,
)

In [ ]:
cb = CatBoostRegressor(
    loss_function='RMSE',
    **cb_params,
)

In [ ]:
aci_res = aci_ts_regressor_predict(
    model=cb,
    cv=cv,
    train_data=(
        pd.concat([X_train, X_calib]),
        pd.concat([y_train, y_calib]),
    ),
    test_data=(X_test, y_test),
    update_calibration=True,
    gamma=0.05,
    forecast_horizon=1,
    alpha_list=[1 - 0.95],
)

In [ ]:
df_eval = pd.DataFrame({
    'y_test': y_test,
    'y_pred_low': aci_res[0]['y_pis'][:, 0, 0],
    'y_pred_hig': aci_res[0]['y_pis'][:, 1, 0],
})

df_eval['is_covered'] = df_eval['y_test'].ge(df_eval['y_pred_low']) & df_eval['y_test'].le(df_eval['y_pred_hig'])

In [ ]:
# df_eval.to_parquet(Path(DATA_OUT, 'eval_aci_5_ahead.parquet'), engine='pyarrow')

In [ ]:
# df_eval = pd.read_parquet(Path(DATA_OUT, 'eval_aci_5_ahead.parquet'), engine='pyarrow')
df_eval['width'] = df_eval['y_pred_hig'] - df_eval['y_pred_low']
df_eval['width_category'] = pd.cut(df_eval['width'], bins=9)
df_eval['y_test_category'] = pd.cut(df_eval['y_test'], bins=9)
df_eval['is_covered'] = df_eval['y_test'].ge(df_eval['y_pred_low']) & df_eval['y_test'].le(df_eval['y_pred_hig'])

In [ ]:
df_agg_yc = df_eval.groupby('y_test_category', observed=False).agg(
    n_samples=('width','count'),
    n_covered=('is_covered','sum'),
    mean_width=('width', 'mean'),
)

df_agg_yc['perc_covered'] = np.round(
    100 * df_agg_yc['n_covered'].div(df_agg_yc['n_samples']),
    1,
)

In [ ]:
df_agg_yc

In [ ]:
df_eval = pd.DataFrame({
    'y_pred': aci_res[0]['y_pred'],
    'y_pi_low': aci_res[0]['y_pis'][:, 0, 0],
    'y_pi_upp': aci_res[0]['y_pis'][:, 1, 0],
    'y_true': y_test,
})

df_eval['is_scint'] = df_eval['y_true'].ge(LW_S4_THRESHOLD)
df_eval['is_scint_covered'] = df_eval['is_scint'] & (
    df_eval['y_pi_low'].le(df_eval['y_true'])) & (
    df_eval['y_true'].le(df_eval['y_pi_upp'])
)
df_eval['is_scint_covered_upward'] = df_eval['is_scint'] & (
    df_eval['y_true'].le(df_eval['y_pi_upp'])
)

In [ ]:
perc_scint = df_eval['is_scint'].eq(True).sum() / df_eval.shape[0]
perc_scint_covered = df_eval['is_scint_covered'].eq(True).sum() / df_eval['is_scint'].eq(True).sum()
perc_scint_covered_upward = df_eval['is_scint_covered_upward'].eq(True).sum() / df_eval['is_scint'].eq(True).sum()

In [ ]:
print(
    f'Scintillation happens {perc_scint:.1%} of the time. Scintillation is covered {perc_scint_covered:.1%} of the time (upward: {perc_scint_covered_upward:.1%})'
)

In [ ]:
plot_dict = aci_res

fig, ax = plt.subplots(figsize=(16, 6))
ax.set_ylabel("<S4>", fontsize=16)
ax.plot(y_test, lw=1, ls='-', label="Actual (test)", c="tab:orange")
# ax.plot(
#     y_test.index,
#     plot_dict[0]["y_pred"],
#     lw=1,
#     ls=':',
#     c="tab:blue",
#     label="Forecast",
# )

for i, result_ in enumerate(plot_dict):
    y_pis = result_["y_pis"]
    color = plt.cm.Blues(1 - i/len(plot_dict))
    ax.fill_between(
        y_test.index,
        y_pis[:, 0, 0],
        y_pis[:, 1, 0],
        alpha=0.3,
        color=color,
        label=f"{1 - result_['alpha']:.0%} CL (cover. {result_['coverage']:.0%} – mean width {result_['mean_width']:.2f} – CWC {result_['cwc']:.2f})",
    )

ax.set_title('CatBoost + ACI', fontweight="bold", size=18)
plt.xticks(rotation=45)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%m-%Y %H:%M'))
ax.yaxis.grid(True, color='k', linewidth=0.2, alpha=0.4)
[ax.spines[s].set_visible(False) for s in ax.spines]
ax.legend(loc='upper right', frameon=True, facecolor='white', edgecolor='none')
# ax.set_xlim(y_test.index[0], y_test.index[-1])
ax.set_xlim(y_test.loc['2023-03-19 23'].index[0], y_test.loc['2023-03-20 02'].index[-1])
# ax.set_ylim(0, 0.85)

# plt.savefig('aci_.png', dpi=800, bbox_inches='tight')
plt.show()

## CatBoost + [CQR](https://github.com/superlinear-ai/conformal-tights?tab=readme-ov-file#predicting-quantiles) (conformal tights)

In [ ]:
from conformal_tights import ConformalCoherentQuantileRegressor

In [ ]:
cb = CatBoostRegressor(
    loss_function='RMSE',
    **cb_params
)

In [ ]:
cqr_cb = ConformalCoherentQuantileRegressor(estimator=cb).fit(
    pd.concat([X_train, X_calib]),
    pd.concat([y_train, y_calib]),
)

In [ ]:
y_pred_test_quant = cqr_cb.predict_quantiles(
    X_test, quantiles=(0.0250, 0.500, 0.975),
)

In [ ]:
df_eval = pd.DataFrame()

df_eval['y_test'] = y_test
df_eval['y_pred_low'] = y_pred_test_quant[0.025].clip(0)
df_eval['y_pred_hig'] = np.where(
    y_pred_test_quant[0.975].ge(df_eval['y_pred_low']),
    y_pred_test_quant[0.975],
    df_eval['y_pred_low'],
)
df_eval['is_covered'] = df_eval['y_test'].ge(df_eval['y_pred_low']) & df_eval['y_test'].le(df_eval['y_pred_hig'])

# df_eval.to_parquet(Path(DATA_OUT, 'eval_cqr_ct_5_ahead.parquet'), engine='pyarrow')

df_eval['width'] = df_eval['y_pred_hig'] - df_eval['y_pred_low']
df_eval['width_category'] = pd.cut(df_eval['width'], bins=9)
df_eval['y_test_category'] = pd.cut(df_eval['y_test'], bins=9)

In [ ]:
df_agg_yc = df_eval.groupby('y_test_category', observed=False).agg(
    n_samples=('width','count'),
    n_covered=('is_covered','sum'),
    mean_width=('width', 'mean'),
)

df_agg_yc['perc_covered'] = np.round(
    100 * df_agg_yc['n_covered'].div(df_agg_yc['n_samples']),
    1,
)

In [ ]:
df_agg_yc

In [ ]:
rolling_cov = []
window = 180

for i in range(window, y_test.shape[0], 1):
    rolling_cov.append(
        (
            (y_pred_test_quant[0.025].values[i-window:i] <= y_test[i-window:i]) &
            (y_test[i-window:i] <= y_pred_test_quant[0.975].values[i-window:i])
        ).sum() / window
    )

In [ ]:
plt.figure(figsize=(10, 5))
plt.ylabel(f"Rolling coverage [{window} hours]")

plt.plot(
    y_test[window:].index,
    rolling_cov,
    linestyle='--', color='tab:blue', alpha=0.8
)

plt.show()

In [ ]:
dt_1, dt_2 = '2023-03-26 00', '2023-03-26 01'

plt.figure(figsize=(20, 10))
plt.plot(y_test.loc[dt_1:dt_2], color='tab:blue')

plt.fill_between(
    x=y_pred_test_quant.loc[dt_1:dt_2].index,
    y1=y_pred_test_quant.loc[dt_1:dt_2,0.975],
    y2=y_pred_test_quant.loc[dt_1:dt_2,0.025],
    color='gray',
    alpha=0.2,
    label='Prediction Interval',
)

plt.show()